In [1]:
import torch
import plotly.express as px
import pandas as pd
import numpy as np
from typing import Optional
import pathlib

In [2]:
from epsilon_transformers.persistence import Persister
from epsilon_transformers.process.processes import PROCESS_REGISTRY

/opt/anaconda3/envs/epstrans/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/opt/anaconda3/envs/epstrans/lib/python3.14/site-packages/pydantic/_internal/_generate_schema.py:2249: UnsupportedFieldAttributeWarning: The 'repr' attribute with value False was provided to the `Field()` function, which has no effect in the context it was used. 'repr' is field-specific metadata, and can only be attached to a model field using `Annotated` metadata or by assignment. This may have happened because an `Annotated` type alias using the `type` statement was used, or if the `Field()` function was attached to a single member of a union type.
  warnings.warn(
/opt/anaconda3/envs/epstrans/lib/python3.14/site-packages/pydantic/_internal/_generate_schema.py:2249: UnsupportedFieldAttributeWarning: The 'frozen' attribute wit

In [3]:
from torch import device

from epsilon_transformers import persistence



checkpoint_dir = pathlib.Path("/Users/sbhandari/Documents/GitHub/epsilon-transformers/models/2_layer_0.05_0.85_lr_1e-2_1b")
if torch.cuda.is_available():
    device = device("cuda:0")
elif torch.backends.mps.is_available():
    device = device("mps")
else:
    device = device("cpu")

persister = Persister(checkpoint_dir)
model=persister.load_final_model(device=device)
train_config = persister.load_training_config()
#model=persister.load_model("path")
model.eval()

[Persister] Found 2 checkpoints in /Users/sbhandari/Documents/GitHub/epsilon-transformers/models/2_layer_0.05_0.85_lr_1e-2_1b
[Persister] Found 2 checkpoints in /Users/sbhandari/Documents/GitHub/epsilon-transformers/models/2_layer_0.05_0.85_lr_1e-2_1b
[Persister] Found 2 checkpoints in /Users/sbhandari/Documents/GitHub/epsilon-transformers/models/2_layer_0.05_0.85_lr_1e-2_1b


HookedTransformer(
  (embed): Embed()
  (hook_embed): HookPoint()
  (pos_embed): PosEmbed()
  (hook_pos_embed): HookPoint()
  (blocks): ModuleList(
    (0-1): 2 x TransformerBlock(
      (ln1): LayerNorm(
        (hook_scale): HookPoint()
        (hook_normalized): HookPoint()
      )
      (ln2): LayerNorm(
        (hook_scale): HookPoint()
        (hook_normalized): HookPoint()
      )
      (attn): Attention(
        (hook_k): HookPoint()
        (hook_q): HookPoint()
        (hook_v): HookPoint()
        (hook_z): HookPoint()
        (hook_attn_scores): HookPoint()
        (hook_pattern): HookPoint()
        (hook_result): HookPoint()
      )
      (mlp): MLP(
        (hook_pre): HookPoint()
        (hook_post): HookPoint()
      )
      (hook_attn_in): HookPoint()
      (hook_q_input): HookPoint()
      (hook_k_input): HookPoint()
      (hook_v_input): HookPoint()
      (hook_mlp_in): HookPoint()
      (hook_attn_out): HookPoint()
      (hook_mlp_out): HookPoint()
      (hook_resi

In [4]:
process_name = 'Mess3'
process_params ={
    "x": 0.05,
    "a": 0.85
}
seq_len = 10
num_tokens_vocab = 3
if process_name in PROCESS_REGISTRY:
    process=PROCESS_REGISTRY[process_name](**process_params)

In [5]:
history=process.generate_process_history(total_length=10)
input_seq=torch.tensor([history.symbols],dtype=torch.long,device=device)
print(input_seq)


tensor([[2, 2, 1, 2, 2, 2, 0, 2, 2, 2]], device='mps:0')


In [6]:
def get_attention_hooks(model):
    return [name for name,_ in model.named_modeules() if "attn.hook_pattern" in name]

In [7]:
from pandas.core.computation.ops import Op


def plot_attention_heatmap(
    cache,
    layer_idx:int,
    head_idx: Optional[int]=None,
    tokens:Optional[list[str]]=None
):
    hook_name=f"blocks.{layer_idx}.attn.hook_pattern"
    if hook_name not in cache:
        print("error: hook not found in cache")
        return
    #pattern has shape [batch, n_heads, query_pos, key_pos]    
    pattern=cache[hook_name][0].detach().cpu().numpy()
    n_heads=pattern.shape[0]
    seq_len=pattern.shape[1]  

    if tokens is None:
        x_labels=[f"Key{i}" for i in range(seq_len)] 
        y_labels=[f"Query{i}" for i in range(seq_len)]
    else:
        x_labels = [f"{t} (K{i})" for i, t in enumerate(tokens)]
        y_labels = [f"{t} (Q{i})" for i, t in enumerate(tokens)]

    if head_idx is not None:
        print(f"Plotting Layer {layer_idx}, Head {head_idx}")
        fig = px.imshow(
            pattern[head_idx],
            labels=dict(x="Key (Source)", y="Query (Destination)", color="Attention"),
            x=x_labels,
            y=y_labels,
            title=f"Attention Pattern: Layer {layer_idx}, Head {head_idx}",
            color_continuous_scale="Viridis",
            range_color=[0, 1] # Attention sums to 1
        )
        fig.update_layout(width=600, height=600)
    else:
        # All Heads Faceted Plot
        print(f"Plotting All Heads for Layer {layer_idx}")
        fig = px.imshow(
            pattern,
            labels=dict(x="Key", y="Query", color="Attn", facet_col="Head"),
            x=x_labels,
            y=y_labels,
            facet_col=0, # Facet over the first dimension (heads)
            facet_col_wrap=min(n_heads, 4), # Wrap after 4 heads
            title=f"Attention Patterns: Layer {layer_idx} (All Heads)",
            color_continuous_scale="Viridis",
            range_color=[0, 1]
        )
        fig.update_layout(height=400 * ((n_heads + 3) // 4), width=1200)

    fig.show()

In [8]:
logits, cache = model.run_with_cache(input_seq)

In [9]:
token_labels = [str(t.item()) for t in input_seq[0]]

In [16]:
plot_attention_heatmap(cache, layer_idx=0, head_idx=0, tokens=token_labels)

Plotting Layer 0, Head 0
